# Phase-1: ICA (Incremental Capacity Analysis)

**Goal:** Understand and visualize ICA (dQ/dV) analysis step-by-step.

**ICA Background:**
- ICA = dQ/dV (differential capacity)
- Peaks in ICA curve correspond to phase transitions in electrode materials
- Peak shifts indicate degradation mechanisms
- Critical physics-based feature for SOH estimation

**Steps:**
1. Select sample cycles (early, middle, late)
2. Compute Q(t) from current integration
3. Smooth voltage and capacity curves
4. Compute dQ/dV numerically
5. Detect and analyze peaks
6. Compare ICA evolution across aging

In [ ]:
# --- Portable setup (Windows / macOS / Linux) — do not hardcode paths ---
import importlib.util
from pathlib import Path

def _load_notebook_setup():
    cwd = Path.cwd().resolve()
    candidates = [
        cwd / "notebooks" / "notebook_setup.py",
        cwd / "notebook_setup.py",
    ]
    if cwd.name == "notebooks":
        candidates.insert(0, cwd / "notebook_setup.py")
    for fp in candidates:
        if fp.is_file():
            spec = importlib.util.spec_from_file_location("_notebook_setup", fp)
            mod = importlib.util.module_from_spec(spec)
            assert spec.loader is not None
            spec.loader.exec_module(mod)
            return mod
    raise FileNotFoundError(
        "Missing notebooks/notebook_setup.py. Unzip the full project, open a terminal, "
        "cd into the repo root, then run: jupyter notebook"
    )

_ns = _load_notebook_setup()
PROJECT_ROOT = _ns.setup()
print(f"PROJECT_ROOT = {PROJECT_ROOT}")


import sys
import os



import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter, find_peaks

from utils import load_config, plot_style_setup
from data_loader import load_battery_data
from cycle_segmentation import segment_and_filter_cycles

plot_style_setup()
config = load_config('config.yaml')

print("✓ Imports complete")

## 1. Load Filtered Cycles

In [ ]:
# Load battery data
loader = load_battery_data(config)
filtered_cycles = segment_and_filter_cycles(loader, config)

print(f"\nTotal filtered cycles: {len(filtered_cycles)}")

## 2. Select Sample Cycles

In [ ]:
# Early, middle, late cycles
n = len(filtered_cycles)
sample_indices = [0, n//2, n-1]
sample_cycles = [filtered_cycles[i] for i in sample_indices]
sample_labels = [f'Cycle {i} (Early)', f'Cycle {n//2} (Middle)', f'Cycle {n-1} (Late)']
colors = ['green', 'orange', 'red']

print(f"\nSelected cycles: {sample_indices}")

## 3. ICA Computation Step-by-Step (Single Cycle)

In [ ]:
# Demonstrate ICA computation on first cycle
cycle = sample_cycles[0]

print(f"\n=== ICA Computation for {sample_labels[0]} ===")

# Step 1: Compute capacity Q(t) = ∫ I dt
time = cycle.time
current = cycle.current
voltage = cycle.voltage

dt = np.diff(time, prepend=time[0])
capacity = np.cumsum(np.abs(current) * dt) / 3600.0  # Ah

print(f"  Step 1: Computed Q(t) from current integration")
print(f"          Total capacity: {capacity[-1]:.4f} Ah")

# Step 2: Filter data in voltage range of interest
v_min, v_max = config['features']['ica']['voltage_range']
mask = (voltage >= v_min) & (voltage <= v_max)

voltage_filtered = voltage[mask]
capacity_filtered = capacity[mask]

print(f"  Step 2: Filtered data in voltage range [{v_min}, {v_max}] V")
print(f"          Data points: {len(voltage_filtered)}")

# Step 3: Sort by voltage (decreasing for discharge)
sort_idx = np.argsort(voltage_filtered)
voltage_sorted = voltage_filtered[sort_idx]
capacity_sorted = capacity_filtered[sort_idx]

print(f"  Step 3: Sorted by voltage")

# Step 4: Smooth capacity curve
window = config['features']['ica']['window_length']
polyorder = config['features']['ica']['polyorder']

capacity_smooth = savgol_filter(capacity_sorted, window_length=window, polyorder=polyorder)

print(f"  Step 4: Applied Savitzky-Golay filter (window={window}, order={polyorder})")

# Step 5: Compute dQ/dV
dqdv = np.gradient(capacity_smooth, voltage_sorted)

print(f"  Step 5: Computed dQ/dV using numerical gradient")

# Step 6: Smooth dQ/dV curve
dqdv_smooth = savgol_filter(dqdv, window_length=window, polyorder=polyorder)

print(f"  Step 6: Smoothed dQ/dV curve")

# Step 7: Find peaks
peaks, properties = find_peaks(
    np.abs(dqdv_smooth),
    prominence=config['features']['ica']['peak_detection']['prominence'],
    width=config['features']['ica']['peak_detection']['width']
)

print(f"  Step 7: Detected {len(peaks)} peaks")

if len(peaks) > 0:
    for i, peak_idx in enumerate(peaks[:3]):  # Top 3
        print(f"          Peak {i+1}: V = {voltage_sorted[peak_idx]:.3f} V, dQ/dV = {dqdv_smooth[peak_idx]:.4f}")

## 4. Visualize ICA Computation Process

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Plot 1: Raw discharge curve V(t)
axes[0, 0].plot(time, voltage, linewidth=1)
axes[0, 0].set_xlabel('Time (s)')
axes[0, 0].set_ylabel('Voltage (V)')
axes[0, 0].set_title('Step 1: Voltage vs Time (Raw)')
axes[0, 0].grid(True, alpha=0.3)

# Plot 2: Q vs V (discharge curve)
axes[0, 1].plot(capacity, voltage, linewidth=1, label='Raw')
axes[0, 1].plot(capacity_smooth, voltage_sorted, linewidth=2, label='Smoothed', alpha=0.7)
axes[0, 1].set_xlabel('Capacity (Ah)')
axes[0, 1].set_ylabel('Voltage (V)')
axes[0, 1].set_title('Step 2-4: Capacity vs Voltage (Smoothed)')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Plot 3: dQ/dV curve (raw and smoothed)
axes[1, 0].plot(voltage_sorted, dqdv, linewidth=1, alpha=0.5, label='Raw dQ/dV')
axes[1, 0].plot(voltage_sorted, dqdv_smooth, linewidth=2, label='Smoothed dQ/dV')
axes[1, 0].set_xlabel('Voltage (V)')
axes[1, 0].set_ylabel('dQ/dV (Ah/V)')
axes[1, 0].set_title('Step 5-6: Incremental Capacity Curve')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Plot 4: Peaks detected
axes[1, 1].plot(voltage_sorted, np.abs(dqdv_smooth), linewidth=2, label='|dQ/dV|')
if len(peaks) > 0:
    axes[1, 1].plot(voltage_sorted[peaks], np.abs(dqdv_smooth[peaks]), 'ro', markersize=10, label='Detected Peaks')
    for peak_idx in peaks:
        axes[1, 1].annotate(
            f'{voltage_sorted[peak_idx]:.2f}V',
            xy=(voltage_sorted[peak_idx], np.abs(dqdv_smooth[peak_idx])),
            xytext=(5, 5), textcoords='offset points', fontsize=9
        )
axes[1, 1].set_xlabel('Voltage (V)')
axes[1, 1].set_ylabel('|dQ/dV| (Ah/V)')
axes[1, 1].set_title('Step 7: Peak Detection')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.suptitle(f'ICA Computation Process - {sample_labels[0]}', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Compare ICA Across Aging (Early, Middle, Late)

In [ ]:
def compute_ica(cycle, config):
    """Compute ICA for a cycle."""
    time = cycle.time
    current = cycle.current
    voltage = cycle.voltage
    
    # Compute capacity
    dt = np.diff(time, prepend=time[0])
    capacity = np.cumsum(np.abs(current) * dt) / 3600.0
    
    # Filter voltage range
    v_min, v_max = config['features']['ica']['voltage_range']
    mask = (voltage >= v_min) & (voltage <= v_max)
    
    voltage_filtered = voltage[mask]
    capacity_filtered = capacity[mask]
    
    # Sort
    sort_idx = np.argsort(voltage_filtered)
    voltage_sorted = voltage_filtered[sort_idx]
    capacity_sorted = capacity_filtered[sort_idx]
    
    # Smooth
    window = config['features']['ica']['window_length']
    polyorder = config['features']['ica']['polyorder']
    capacity_smooth = savgol_filter(capacity_sorted, window_length=window, polyorder=polyorder)
    
    # Compute dQ/dV
    dqdv = np.gradient(capacity_smooth, voltage_sorted)
    dqdv_smooth = savgol_filter(dqdv, window_length=window, polyorder=polyorder)
    
    # Find peaks
    peaks, _ = find_peaks(
        np.abs(dqdv_smooth),
        prominence=config['features']['ica']['peak_detection']['prominence'],
        width=config['features']['ica']['peak_detection']['width']
    )
    
    return voltage_sorted, dqdv_smooth, peaks

# Compute ICA for all sample cycles
ica_results = []
for cycle in sample_cycles:
    v, dqdv, peaks = compute_ica(cycle, config)
    ica_results.append((v, dqdv, peaks))

# Plot comparison
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: ICA curves overlay
for (v, dqdv, peaks), label, color in zip(ica_results, sample_labels, colors):
    axes[0].plot(v, dqdv, label=label, color=color, linewidth=2)
    if len(peaks) > 0:
        axes[0].plot(v[peaks], dqdv[peaks], 'o', color=color, markersize=8)

axes[0].set_xlabel('Voltage (V)', fontsize=12)
axes[0].set_ylabel('dQ/dV (Ah/V)', fontsize=12)
axes[0].set_title('ICA Evolution: Peak Shifts During Aging', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# Plot 2: Absolute value for clarity
for (v, dqdv, peaks), label, color in zip(ica_results, sample_labels, colors):
    axes[1].plot(v, np.abs(dqdv), label=label, color=color, linewidth=2)
    if len(peaks) > 0:
        axes[1].plot(v[peaks], np.abs(dqdv[peaks]), 'o', color=color, markersize=8)

axes[1].set_xlabel('Voltage (V)', fontsize=12)
axes[1].set_ylabel('|dQ/dV| (Ah/V)', fontsize=12)
axes[1].set_title('ICA Evolution: Absolute Value', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Analyze Peak Shifts

In [ ]:
# Extract peak positions for all cycles
all_peak_positions = []

for i, cycle in enumerate(filtered_cycles):
    v, dqdv, peaks = compute_ica(cycle, config)
    if len(peaks) > 0:
        # Get primary peak (highest magnitude)
        primary_peak_idx = peaks[np.argmax(np.abs(dqdv[peaks]))]
        primary_peak_voltage = v[primary_peak_idx]
        all_peak_positions.append(primary_peak_voltage)
    else:
        all_peak_positions.append(np.nan)

# Plot peak shift over cycles
fig, ax = plt.subplots(figsize=(12, 6))

valid_indices = [i for i, p in enumerate(all_peak_positions) if not np.isnan(p)]
valid_peaks = [all_peak_positions[i] for i in valid_indices]

ax.plot(valid_indices, valid_peaks, 'o-', markersize=4, linewidth=2)
ax.set_xlabel('Cycle Number', fontsize=12)
ax.set_ylabel('Primary ICA Peak Voltage (V)', fontsize=12)
ax.set_title('ICA Peak Shift During Battery Aging', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)

if len(valid_peaks) > 0:
    initial_peak = valid_peaks[0]
    final_peak = valid_peaks[-1]
    peak_shift = final_peak - initial_peak
    
    ax.axhline(y=initial_peak, color='g', linestyle='--', alpha=0.5, label=f'Initial: {initial_peak:.3f} V')
    ax.axhline(y=final_peak, color='r', linestyle='--', alpha=0.5, label=f'Final: {final_peak:.3f} V')
    ax.legend()
    
    print(f"\nICA Peak Analysis:")
    print(f"  Initial peak position: {initial_peak:.3f} V")
    print(f"  Final peak position: {final_peak:.3f} V")
    print(f"  Total peak shift: {peak_shift:.3f} V ({peak_shift/initial_peak*100:.2f}%)")
    print(f"\n  Peak shift is a strong indicator of battery degradation!")

plt.tight_layout()
plt.show()

## Summary

**Key Findings:**
1. ICA (dQ/dV) reveals characteristic peaks related to electrode phase transitions
2. Peaks shift and change magnitude as battery ages
3. Peak shift is a physics-based indicator of degradation mechanisms
4. Smoothing is critical for reliable peak detection
5. ICA features provide complementary information to capacity fade

**Physics Insight:**
- Peak shifts indicate changes in electrode material structure
- Loss of lithium inventory, loss of active material
- Different degradation modes have different ICA signatures

**Next Steps:**
- Extract ICA features for all cycles (automated in feature_extraction.py)
- Use ICA peaks as input features for ML model
- Compare model performance with/without ICA features